# Shoreline on Colab

Train a small model to answer **Land or Water?** from a latitude and longitude.

1. **Runtime → Change runtime type → T4 GPU.**
2. Put this repo on the Colab machine. Upload the `shoreline` folder into the file browser, or keep it in Google Drive under My Drive.
3. Run the cells from the top. Twenty steps on the default 0.5B model takes a few minutes on a free T4.


In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit('No GPU is visible. Use Runtime → Change runtime type → T4 GPU, then rerun this cell.')
name = torch.cuda.get_device_name(0)
free, total = torch.cuda.mem_get_info()
print(f'{name}  {free / 1024**3:.1f} GB free of {total / 1024**3:.1f} GB')


In [ ]:
from pathlib import Path
import sys

def looks_like_repo(path: Path) -> bool:
    return (path / 'shoreline' / 'data' / 'land_mask.npz').is_file()

def search(root: Path, depth: int) -> Path | None:
    if not root.is_dir():
        return None
    if looks_like_repo(root):
        return root
    if depth <= 0:
        return None
    try:
        children = list(root.iterdir())
    except OSError:
        return None
    for child in children:
        if child.is_dir() and not child.name.startswith('.'):
            found = search(child, depth - 1)
            if found is not None:
                return found
    return None

roots = [Path.cwd(), Path('/content')]
drive = Path('/content/drive/MyDrive')
if not drive.is_dir():
    try:
        from google.colab import drive as colab_drive
        colab_drive.mount('/content/drive')
    except Exception as exc:
        print('Drive was not mounted.', exc)
if drive.is_dir():
    roots.append(drive)

ROOT = None
for root in roots:
    ROOT = search(root, 3 if root == drive else 2)
    if ROOT is not None:
        break
if ROOT is None:
    raise FileNotFoundError('Upload the Shoreline folder into /content, or place it in My Drive, then rerun this cell.')
sys.path.insert(0, str(ROOT))
print(ROOT)


In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'peft', 'transformers', 'accelerate', 'pillow', 'huggingface_hub'])
subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'])


In [ ]:
from IPython.display import Image, display
from shoreline.sim import ShoreSim

sim = ShoreSim()
place = sim.reset(latitude=48.86, longitude=2.35)
print(place.prompt)
display(Image(data=place.image_png))
print('reward for saying land:', sim.step('land').reward)

land_cells = float(sim.world.land.mean())
print(f'land cells: {land_cells:.1%}')
print(f'always answering water scores about {1 - land_cells:.1%} on a uniform sample')


## GRPO

Each step asks one coordinate four times. The four rewards become advantages inside that group, so a tied group does not move the model. Training samples land and water evenly. A checkpoint is written every 5 steps under `/content/shoreline-adapter`. Set `HF_TOKEN` before this cell to push each checkpoint to a private Hugging Face repo.


In [ ]:
from shoreline.grpo import train_survey

history = train_survey(steps=20, group_size=4, output_dir='/content/shoreline-adapter', save_steps=5)
print('mean reward over the last 5 steps:', sum(history[-5:]) / len(history[-5:]))
